# Welcome to Colab!

In [ ]:
!pip install -qU transformers accelerate sentence-transformers faiss-cpu rank_bm25

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 62.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 570.8/570.8 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 35.5 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import torch
import faiss
import pickle
import json
import numpy as np
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoModelForCausalLM, AutoTokenizer

# ==========================================
# 1. LOAD RAG SYSTEM TỪ GOOGLE DRIVE
# ==========================================
save_path = '/content/drive/MyDrive/Vector_DB_Ultimate'
print("⏳ Đang tải cơ sở dữ liệu Vector và BM25...")
index = faiss.read_index(f"{save_path}/faiss_bge_m3.index")
with open(f"{save_path}/bm25_model.pkl", "rb") as f:
    bm25 = pickle.load(f)
with open(f"{save_path}/chunk_data.json", "r", encoding="utf-8") as f:
    data = json.load(f)
    chunk_ids = data["chunk_ids"]
    chunk_texts_with_meta = data["chunk_texts_with_meta"]

print("⏳ Đang tải BGE-M3 và Reranker...")
embedder = SentenceTransformer('BAAI/bge-m3')
reranker = CrossEncoder('BAAI/bge-reranker-v2-m3', max_length=512)

# ==========================================
# 2. LOAD LLM (QWEN) TỪ HUGGING FACE
# ==========================================
print("⏳ Đang tải mô hình LLM (Qwen)...")
# Bạn có thể đổi tên model ở đây thành phiên bản Qwen bạn muốn
# Ví dụ: "Qwen/Qwen2.5-3B-Instruct" rất phù hợp để chạy mượt trên Colab T4
model_id = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)
# Load model bằng float16 để tiết kiệm VRAM
llm = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)
print("✅ Hoàn tất! Hệ thống đã sẵn sàng.\n")
print("="*60)

# ==========================================
# 3. VÒNG LẶP CHATBOT (TERMINAL)
# ==========================================
def search_law(query, top_k=3):
    # Hybrid Search -> Rerank (Giữ nguyên logic cực mạnh từ trước)
    tokenized_query = query.lower().split(" ")
    bm25_scores = bm25.get_scores(tokenized_query)
    top_bm25 = np.argsort(bm25_scores)[::-1][:15]

    query_emb = embedder.encode([query], normalize_embeddings=True)
    _, top_faiss = index.search(query_emb, 15)
    top_faiss = top_faiss[0]

    combined_indices = list(set(top_bm25).union(set(top_faiss)))
    candidate_texts = [chunk_texts_with_meta[idx] for idx in combined_indices]

    pairs = [[query, text] for text in candidate_texts]
    rerank_scores = reranker.predict(pairs)

    scored_candidates = list(zip(candidate_texts, rerank_scores))
    scored_candidates.sort(key=lambda x: x[1], reverse=True)

    return [x[0] for x in scored_candidates[:top_k]]

# Bắt đầu vòng lặp chat
print("🤖 Bắt đầu chat! (Nhập 'exit' hoặc 'quit' để thoát)")
while True:
    query = input("\n👤 Bạn: ")
    if query.lower() in ['exit', 'quit', 'q']:
        print("🤖 Bot: Hẹn gặp lại!")
        break
    if not query.strip():
        continue

    # Bước A: Truy xuất dữ liệu
    retrieved_chunks = search_law(query)
    context = "\n\n".join(retrieved_chunks)

    # Bước B: Xây dựng Prompt cho Qwen
    system_prompt = "Bạn là một trợ lý ảo chuyên tư vấn pháp luật giao thông đường bộ Việt Nam. Hãy dựa vào văn bản luật được cung cấp dưới đây để trả lời câu hỏi của người dùng một cách chính xác, ngắn gọn và lịch sự."
    user_prompt = f"VĂN BẢN LUẬT:\n{context}\n\nCÂU HỎI:\n{query}"

    # Sử dụng Chat Template chuẩn của Qwen
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]
    text_input = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = tokenizer([text_input], return_tensors="pt").to(llm.device)

    # Bước C: Sinh câu trả lời
    generated_ids = llm.generate(
        **model_inputs,
        max_new_tokens=512,
        temperature=0.3, # Để nhiệt độ thấp để bot không bịa thông tin
        do_sample=True
    )

    # Cắt bỏ phần prompt đầu vào để chỉ lấy câu trả lời
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

    print(f"\n🤖 Qwen: {response}")
    print("\n" + "-"*40)
    print(f"📚 Căn cứ tham khảo:\n{context}")
    print("-"*40)

⏳ Đang tải cơ sở dữ liệu Vector và BM25...
⏳ Đang tải BGE-M3 và Reranker...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

✅ Hoàn tất! Hệ thống đã sẵn sàng.

🤖 Bắt đầu chat! (Nhập 'exit' hoặc 'quit' để thoát)

👤 Bạn: ) Điều khiển xe chạy quá tốc độ quy định từ 05 km/h đến dưới 10 km/h bị phạt bao nhiêu tiền

🤖 Qwen: Theo văn bản luật được cung cấp, việc điều khiển xe chạy quá tốc độ quy định từ 5 km/h đến dưới 10 km/h sẽ bị phạt tiền từ 400.000 đồng đến 600.000 đồng. Đây là mức phạt áp dụng cho người điều khiển xe mô tô, xe gắn máy, các loại xe tương tự xe mô tô và xe gắn máy.

----------------------------------------
📚 Căn cứ tham khảo:
[Chương II, Điều 8, Khoản 3, Điểm a] Điều 8. Xử phạt người điều khiển xe máy chuyên dùng vi phạm quy tắc giao thông đường bộ 3. Phạt tiền từ 800.000 đồng đến 1.000.000 đồng đối với người điều khiển xe thực hiện một trong các hành vi vi phạm sau đây: a) Chạy quá tốc độ quy định từ 05 km/h đến dưới 10 km/h;

[Chương II, Điều 6, Khoản 3, Điểm a] Điều 6. Xử phạt, trừ điểm giấy phép lái xe của người điều khiển xe ô tô, xe chở người bốn bánh có gắn động cơ, xe chở hàng bốn bánh 